In [ ]:
import numpy as np
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

from copy import copy

from config import FOCUS_REGION

# Creation of a new tagging region file

For tagging, WAM2layers relies on a netCDF file spanning the tracking region. The only data variable is "source_region", which is 1 at all grid points for which the precipitation is supposed to be tracked.\
Here I create a new file, which includes the expanded source region. For reference, the extent of the Eiffel test case domain and tagging mask:

In [ ]:
ds_region = xr.open_dataset("data/moisture_tracking/invar/region-eiffel.nc") #old tagging file

fig, ax = plt.subplots(1, 1, figsize=(6,4), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=2)

ax.contourf(ds_region["longitude"], ds_region["latitude"], ds_region["source_region"])

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set(title="WAM2layers Demo Region")

rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], 
                      edgecolor='red', linewidth=2, fill=False, label="Focus region")
ax.add_patch(rectangle)
plt.legend()

plt.show()

## Deterministic

Create new tagging mask from data file and config bounds:

In [ ]:
ds = xr.open_dataset("data/moisture_tracking/BLK_WLT/det/icon_R03B07_tp_original_20210714.nc") #some arbitrary file to get coordinates

mask = (ds["lon"] >= FOCUS_REGION["x0"]) & (ds["lon"] <= FOCUS_REGION["x1"]) & (ds["lat"] >= FOCUS_REGION["y0"]) & (ds["lat"] <= FOCUS_REGION["y1"])
vals = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), dtype=np.float32)
vals[mask.T] = 1.

ds_new_region = xr.Dataset(
    data_vars={
        "source_region": (["latitude", "longitude"], vals)
    },
    coords={
        "longitude": ds["lon"].values,
        "latitude": ds["lat"].values
    }
)

ds_new_region.to_netcdf("data/moisture_tracking/invar/expanded_region.nc") # and save it

The "updated" tagging region now looks like this:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,4), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=2)

# Plot of data A:
ax.contourf(ds_new_region["longitude"], ds_new_region["latitude"], ds_new_region["source_region"])

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set(title="New Tagging Region")

rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], 
                      edgecolor='red', linewidth=2, fill=False, label="Focus region")
ax.add_patch(rectangle)
plt.legend()


plt.show()

## Ensemble

In [ ]:
ds = xr.open_dataset("data/moisture_tracking/BLK_WLT/mem001/icon_R02B06_tp_20210714_mem001.nc") #some arbitrary file to get coordinates

mask = (ds["lon"] >= FOCUS_REGION["x0"]) & (ds["lon"] <= FOCUS_REGION["x1"]) & (ds["lat"] >= FOCUS_REGION["y0"]) & (ds["lat"] <= FOCUS_REGION["y1"])
vals = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), dtype=np.float32)
vals[mask.T] = 1.

ds_new_region = xr.Dataset(
    data_vars={
        "source_region": (["latitude", "longitude"], vals)
    },
    coords={
        "longitude": ds["lon"].values,
        "latitude": ds["lat"].values
    }
)

ds_new_region.to_netcdf("data/moisture_tracking/invar/expanded_region_ens.nc") # and save it

# Dis-aggregating precipitation and converting units

Due to the storage logic, 00UTC contains 21,22,23 and 00UTC. At 21UTC, precipitation is zero and the following time steps contain the accumulated sum of precipitation. Since WAM2layers expects hourly values (for now), I dis-aggregate them here. Also, I convert the units from mm in ICON to m as in ERA5 and that of evaporation from flux to column equivalent. Those are just factors of 1000 and 3.6 (see code). This is done, because I want to bring the data into the same format as ERA5, because that is what my modifications are based upon.

In [ ]:
def diff_within_blocks_vectorized(da, block_size=3):
    """
    Disaggregates the 3h accumulated variables produced by the assimilation system.
    The logic is: [(0),1,2,3], [(3),4,5,6], [(6),7,8,9], ... UTC are blocks. Within each block,
    some variables are accumulated, e.g., total precipitaiton, initialized from zero (values in brackets).
    For example, the hourly precipitation would be 1 UTC - 0 UTC (all zero), 2 UTC - 1 UTC, 
    3 UTC - 2 UTC. Note that this creates an overlap, but we are only processing/loading data on the meaningful
    timesteps.
    """

    # Create groups based on 3h intervals:
    block_labels = np.zeros(len(da["time"]), dtype=int)

    i_block = 0
    for i, dt in enumerate(da.time):
        if dt.dt.hour in [1, 4, 7, 10, 13, 16, 19, 22]:
            i_block += 1
        
        block_labels[i] = i_block
    
    da_with_blocks = da.assign_coords(block=('time', block_labels)) #add block labels as a coordinate
 
    def block_diff(block):
        if len(block.time) > 1:
            shifted = block.shift(time=1).values
            shifted[0] = 0.
            
            return block - shifted
        return block
    
    # Apply the function to each block and concatenate results
    result = da_with_blocks.groupby('block').map(block_diff)
    
    return result.drop_vars('block') #drop variables

This precipitation approach has the issue that I get some minimal negative values. I think this might be an artifact from the interpolation. In the interpolated, but accumulated files, everything still is positive by definition. But maybe the differences don't quite add up anymore. Since it's only minor negative values, it's probably safe to ignore.

In [ ]:
exp_name = "icon_dream"

In [ ]:
ds_tp = xr.open_mfdataset([f"data/moisture_tracking/{exp_name}/icon_R03B07_tp_original_202107{dd:02}.nc" for dd in range(1,16)])

da_diff = diff_within_blocks_vectorized(ds_tp["TOT_PREC"]).compute() # conversion from mm to m happens in w2l

for dd in range(1,16):
    print(dd)
    time_slice = slice(np.datetime64(f"2021-07-{dd:02}T00"), np.datetime64(f"2021-07-{dd:02}T23"))
    _ = da_diff.sel(time=time_slice)
    _.where(_ > 0., other=0.).to_netcdf(f"data/moisture_tracking/{exp_name}/icon_R03B07_tp_202107{dd:02}.nc")